# S07 · Chunked prefill: choosing the token budget

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 9. Sources: Sarathi-Serve (arXiv:2403.02310); vLLM V1 scheduler.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Sweep the chunked-prefill token budget and watch TTFT and p99 ITL trade off.

## Theory and equations
Each iteration processes at most $\tau$ tokens. A $T_p$-token prompt needs
$\lceil T_p/(\tau - B_{\text{dec}})\rceil$ iterations; iteration time grows with $\tau$.

## Diagram
```
tau small: many short iterations -> smooth ITL, slower prefill (higher TTFT)
tau large: few long iterations   -> faster prefill, longer gaps between decode tokens
```

## Implementation

In [ ]:
from inference_lab import scheduler as S
trace = S.poisson_trace(150, rate=15.0, seed=2, prompt=(512, 4096))

## Experiment (*theoretical*, roofline-timed)

In [ ]:
budgets = [128, 256, 512, 1024, 2048, 4096]
rs = [S.simulate(trace, "chunked", token_budget=b) for b in budgets]
for b, r in zip(budgets, rs):
    print(f"tau={b:5d}  TTFT mean {r.ttft_mean:6.3f}s  ITL p99 {1e3*r.itl_p99:6.1f} ms  throughput {r.throughput_tok_s:6.0f} tok/s")
fig, ax = plt.subplots(); ax2 = ax.twinx()
ax.plot(budgets, [r.ttft_mean for r in rs], "o-", color=C["compute"]); ax.set_ylabel("mean TTFT (s)", color=C["compute"])
ax2.plot(budgets, [1e3 * r.itl_p99 for r in rs], "s-", color=C["memory"]); ax2.set_ylabel("p99 ITL (ms)", color=C["memory"])
ax.set_xscale("log", base=2); ax.set_xlabel("token budget per iteration"); plt.show()

## Results: separate the two effects (unlimited KV memory)

In [ ]:
rs2 = [S.simulate(trace, "chunked", token_budget=b, num_blocks=10**6) for b in budgets]
for b, r in zip(budgets, rs2):
    print(f"tau={b:5d}  TTFT mean {r.ttft_mean:6.3f}s  ITL p99 {1e3*r.itl_p99:6.1f} ms  preemptions {r.preemptions}")

## Interpretation
TTFT is U-shaped in the budget. A tiny budget makes every iteration pay a full read of the
weights for a handful of prompt tokens, so prefill crawls and the queue grows. A huge budget makes
every iteration long, so ITL rises in proportion and each prompt waits behind other prompts'
chunks. With a realistic KV pool, memory pressure and preemptions add to TTFT on top of that;
the unlimited-memory run isolates the scheduling effect.

## Limitations
Roofline timing; no fixed per-iteration CPU overhead, which would penalise small budgets more.

## Conclusion
Choose the budget from the ITL target first (iteration time grows with it), then check TTFT;
if TTFT is dominated by memory pressure, the fix is more KV capacity, not a different budget.